In [6]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime
import uuid

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

def generate_transaction():
    return {
        "tx_id": str(uuid.uuid4()),
        "user_id": random.randint(1, 60),
        "amount": round(random.uniform(5, 5000), 2),
        "store": random.choice(["Amazon", "Walmart", "Target", "eBay"]),
        "category": random.choice(["electronics", "clothing", "food", "books"]),
        "timestamp": datetime.utcnow().isoformat()
    }

# pętla generująca 1 transakcję na sekundę
while True:
    tx = generate_transaction()
    producer.send('transactions', tx)

    print(f"TX: {tx['tx_id']} | {tx['user_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['category']}")

    time.sleep(1)

Overwriting producer.py


In [7]:
%%file consumer_velocity_anomaly.py
from kafka import KafkaConsumer
import json
from collections import defaultdict, deque
from datetime import datetime

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='velocity-anomaly-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

user_activity = defaultdict(deque)

WINDOW_SECONDS = 60
THRESHOLD = 3

def parse_time(ts):
    return datetime.fromisoformat(ts)

print("Monitoring: >3 transakcji / 60 sekund per user...")

for message in consumer:
    tx = message.value
    user_id = tx["user_id"]
    tx_time = parse_time(tx["timestamp"])

    history = user_activity[user_id]
    history.append(tx_time)

    while history and (tx_time - history[0]).total_seconds() > WINDOW_SECONDS:
        history.popleft()

    if len(history) > THRESHOLD:
        print(f"ALERT: user {user_id} | {len(history)} tx in 60s | last_tx={tx['tx_id']}")

Overwriting consumer_velocity_anomaly.py
